# Working Kaggle Notebook: Virtual Dress Try-On Only — CatVTON

This version is built for **Kaggle Python 3.12 + GPU** and avoids the previous dependency-conflict problem by not downgrading Kaggle’s core stack.

Inputs:
```text
/kaggle/input/person/person.jpg
/kaggle/input/dress/dress.jpg
```

Output:
```text
/kaggle/working/virtual_dress_outputs/virtual_dress_result.png
```

Run **Cell 1** once. It will restart the kernel automatically. After restart, click **Run All**.


## Cell 1 — Safe setup for Kaggle

In [12]:
# =========================
# CELL 1: SAFE KAGGLE SETUP + AUTO RESTART
# =========================
import os
import sys
import subprocess
from pathlib import Path

WORK_DIR = Path("/kaggle/working")
REPO_DIR = WORK_DIR / "CatVTON"
MARKER = WORK_DIR / ".catvton_env_ready_v6"

def run(cmd, cwd=None):
    print("\nRunning:", " ".join(map(str, cmd)))
    subprocess.check_call(list(map(str, cmd)), cwd=cwd)

if not MARKER.exists():
    print("Preparing CatVTON environment for Kaggle Python 3.12...")
    os.environ["HF_HUB_DISABLE_XET"] = "1"

    if not REPO_DIR.exists():
        run(["git", "clone", "https://github.com/Zheng-Chong/CatVTON.git", REPO_DIR])
    else:
        print("CatVTON repository already exists:", REPO_DIR)

    run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip", "setuptools", "wheel"])

    # Clean Pillow install. This fixes: cannot import name '_Ink' from PIL._typing
    run([sys.executable, "-m", "pip", "uninstall", "-y", "Pillow", "PIL"])
    run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "--force-reinstall", "Pillow==10.4.0"])

    # Binary tokenizers wheel first. This avoids Rust build error.
    run([sys.executable, "-m", "pip", "install", "-q", "-U", "--only-binary=tokenizers", "tokenizers>=0.19.1,<0.20.0"])

    # No old numpy/scipy/hub/tqdm/PyYAML downgrades.
    packages = [
        "accelerate==0.31.0",
        "diffusers==0.29.2",
        "transformers==4.44.2",
        "huggingface_hub>=0.34.0,<1.0",
        "safetensors>=0.4.3",
        "opencv-python-headless>=4.10.0.84",
        "matplotlib>=3.9.1",
        "scikit-image>=0.24.0",
        "tqdm>=4.67.1",
        "einops", "fvcore", "iopath", "yacs", "termcolor", "tabulate",
        "omegaconf", "av", "pycocotools", "cloudpickle",
        "scipy>=1.14.1",
        "PyYAML>=6.0.2,<7.0.0",
        "ninja",
    ]
    run([sys.executable, "-m", "pip", "install", "-q", "-U"] + packages)

    try:
        run([sys.executable, "-m", "pip", "install", "-q", "-U", "onnxruntime-gpu"])
    except Exception:
        print("onnxruntime-gpu failed; installing CPU onnxruntime.")
        run([sys.executable, "-m", "pip", "install", "-q", "-U", "onnxruntime"])

    MARKER.write_text("ready")
    print("\nEnvironment installed successfully.")
    print("Restarting Kaggle kernel now. After restart, click Run All.")
    os.kill(os.getpid(), 9)

else:
    print("Environment is already prepared.")
    print("If imports fail, delete /kaggle/working/.catvton_env_ready_v6 and rerun this cell.")


Environment is already prepared.
If imports fail, delete /kaggle/working/.catvton_env_ready_v6 and rerun this cell.


## Cell 2 — Imports and environment check

In [13]:
# =========================
# CELL 2: IMPORTS
# =========================
import os
import sys
import gc
from pathlib import Path

os.environ["HF_HUB_DISABLE_XET"] = "1"

import numpy as np
import torch
import PIL
from PIL import Image
import matplotlib.pyplot as plt

REPO_DIR = Path("/kaggle/working/CatVTON")
assert REPO_DIR.exists(), "CatVTON repository not found. Run Cell 1 first."
sys.path.insert(0, str(REPO_DIR))
os.chdir(str(REPO_DIR))

import diffusers
import transformers
import huggingface_hub
from huggingface_hub import snapshot_download
from diffusers.image_processor import VaeImageProcessor

from model.pipeline import CatVTONPipeline
from model.cloth_masker import AutoMasker
from utils import init_weight_dtype, resize_and_crop, resize_and_padding

print("Pillow:", PIL.__version__)
print("NumPy:", np.__version__)
print("Torch:", torch.__version__)
print("Diffusers:", diffusers.__version__)
print("Transformers:", transformers.__version__)
print("HuggingFace Hub:", huggingface_hub.__version__)

assert torch.cuda.is_available(), "GPU not detected. In Kaggle, set Accelerator = GPU."
device = "cuda"
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


Pillow: 10.4.0
NumPy: 2.4.6
Torch: 2.10.0+cu128
Diffusers: 0.29.2
Transformers: 4.44.2
HuggingFace Hub: 0.36.2
GPU: Tesla T4
VRAM GB: 14.56


## Cell 3 — Load target person and reference dress

Required paths:
```text
/kaggle/input/person/person.jpg
/kaggle/input/dress/dress.jpg
```

In [14]:
# =========================
# CELL 3: LOAD IMAGES
# =========================
import os
PERSON_PATH = "/kaggle/input/datasets/ganapathyg/trynewdata/Dwa.jpeg"
DRESS_PATH = "/kaggle/input/datasets/ganapathyg/trynewdata/dress.jpg"

assert os.path.exists(PERSON_PATH), f"Missing target person image: {PERSON_PATH}"
assert os.path.exists(DRESS_PATH), f"Missing reference dress image: {DRESS_PATH}"

person_image_original = Image.open(PERSON_PATH).convert("RGB")
dress_image_original = Image.open(DRESS_PATH).convert("RGB")

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.imshow(person_image_original)
plt.title("Target person")
plt.axis("off")
plt.subplot(1, 2, 2)
plt.imshow(dress_image_original)
plt.title("Reference dress")
plt.axis("off")
plt.tight_layout()
plt.show()


## Cell 4 — Dress-only configuration

For a full dress, use `CLOTH_TYPE = "overall"`. Do not use `upper` or `lower` for a dress.

In [15]:
# =========================
# CELL 4: DRESS-ONLY CONFIG
# =========================
CLOTH_TYPE = "overall"       # Full dress = overall
WIDTH = 768
HEIGHT = 1024

# If Kaggle T4 gives out-of-memory, use:
# WIDTH = 512
# HEIGHT = 768

NUM_INFERENCE_STEPS = 50
GUIDANCE_SCALE = 2.5
SEED = 42
MIXED_PRECISION = "fp16"

BASE_MODEL_PATH = "runwayml/stable-diffusion-inpainting"
CATVTON_WEIGHTS = "zhengchong/CatVTON"

OUTPUT_DIR = Path("/kaggle/working/virtual_dress_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Try-on mode:", CLOTH_TYPE)
print("Resolution:", WIDTH, "x", HEIGHT)
print("Output folder:", OUTPUT_DIR)


Try-on mode: overall
Resolution: 768 x 1024
Output folder: /kaggle/working/virtual_dress_outputs


## Cell 5 — Load CatVTON model and AutoMasker

In [16]:
# =========================
# CELL 5: LOAD CATVTON
# =========================
gc.collect()
torch.cuda.empty_cache()

print("Downloading CatVTON weights. This can take several minutes...")
repo_path = snapshot_download(repo_id=CATVTON_WEIGHTS)
print("Weights path:", repo_path)

weight_dtype = init_weight_dtype(MIXED_PRECISION)

pipeline = CatVTONPipeline(
    base_ckpt=BASE_MODEL_PATH,
    attn_ckpt=repo_path,
    attn_ckpt_version="mix",
    weight_dtype=weight_dtype,
    use_tf32=True,
    device=device,
    skip_safety_check=True,
)

mask_processor = VaeImageProcessor(
    vae_scale_factor=8,
    do_normalize=False,
    do_binarize=True,
    do_convert_grayscale=True,
)

automasker = AutoMasker(
    densepose_ckpt=os.path.join(repo_path, "DensePose"),
    schp_ckpt=os.path.join(repo_path, "SCHP"),
    device=device,
)

print("CatVTON loaded successfully.")


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Weights path: /root/.cache/huggingface/hub/models--zhengchong--CatVTON/snapshots/2969fcf85fe62f2036605716f0b56f0b81d01d79


An error occurred while trying to fetch runwayml/stable-diffusion-inpainting: runwayml/stable-diffusion-inpainting does not appear to have a file named diffusion_pytorch_model.safetensors.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


CatVTON loaded successfully.


## Cell 6 — Generate virtual dress result

In [17]:
import torch, gc
import accelerate.utils.memory as acc_memory

if not hasattr(acc_memory, "clear_device_cache"):
    def clear_device_cache():
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    acc_memory.clear_device_cache = clear_device_cache

print("PEFT/Accelerate compatibility patch applied.")

PEFT/Accelerate compatibility patch applied.


In [18]:
# =========================
# CELL 6: GENERATE VIRTUAL DRESS
# =========================
gc.collect()
torch.cuda.empty_cache()

person_image = resize_and_crop(person_image_original, (WIDTH, HEIGHT))
dress_image = resize_and_padding(dress_image_original, (WIDTH, HEIGHT))

mask = automasker(person_image, CLOTH_TYPE)["mask"]
mask = mask_processor.blur(mask, blur_factor=9)

generator = torch.Generator(device=device).manual_seed(SEED)

with torch.no_grad():
    result_image = pipeline(
        image=person_image,
        condition_image=dress_image,
        mask=mask,
        num_inference_steps=NUM_INFERENCE_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        generator=generator,
    )[0]

result_path = OUTPUT_DIR / "virtual_dress_result.png"
result_image.save(result_path)

print("Saved final result:", result_path)

plt.figure(figsize=(6, 8))
plt.imshow(result_image)
plt.title("Virtual dress result")
plt.axis("off")
plt.show()


100%|██████████| 50/50 [01:42<00:00,  2.04s/it]


Saved final result: /kaggle/working/virtual_dress_outputs/virtual_dress_result.png


## Cell 7 — Download final result

In [19]:
# =========================
# CELL 7: DOWNLOAD LINK
# =========================
from IPython.display import FileLink, display
display(FileLink(str(OUTPUT_DIR / "virtual_dress_result.png")))


/kaggle/working/virtual_dress_outputs/virtual_dress_result.png

## Notes

If Cell 1 shows a long pip dependency-resolver warning but finishes with **Environment installed successfully**, continue. Kaggle includes many unrelated packages; those warnings usually do not affect CatVTON.

The important check is Cell 2: `Pillow`, `Diffusers`, `Transformers`, and `CatVTONPipeline` must import successfully and GPU must be detected.
